# IPL 2026 Winner Prediction – Analysis Notebook

This notebook walks through the complete analysis pipeline:
1. Data exploration
2. Feature distributions
3. Model comparison
4. Final 2026 predictions

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

from config import FEATURES_CSV, PROCESSED_MATCHES_CSV, RESULTS_DIR, TEAMS

## 1. Load Data

In [ ]:
matches = pd.read_csv(PROCESSED_MATCHES_CSV)
features = pd.read_csv(FEATURES_CSV)
print(f'Matches: {len(matches)} rows')
print(f'Features: {len(features)} rows × {len(features.columns)} cols')
matches.head()

## 2. Matches Per Season

In [ ]:
season_counts = matches.groupby('season').size()
season_counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Matches per IPL Season')
plt.xlabel('Season')
plt.ylabel('Number of Matches')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. All-Time Win Rates

In [ ]:
from src.features.engineer import get_all_time_win_rates
win_rates = get_all_time_win_rates(matches)
active = {k: v for k, v in win_rates.items() if k in TEAMS}
sr = pd.Series(active).sort_values(ascending=False)
sr.plot(kind='barh', color='coral', edgecolor='white')
plt.title('All-time Win Rate by Team')
plt.xlabel('Win Rate')
plt.axvline(0.5, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
sr

## 4. Feature Correlation Heatmap

In [ ]:
from src.models.base_model import FEATURE_COLS
corr = features[FEATURE_COLS + ['team1_won']].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Toss Analysis

In [ ]:
toss_win_rate = matches[matches['toss_winner'] == matches['winner']].shape[0] / len(matches)
print(f'Toss winner also wins match: {toss_win_rate:.1%}')

toss_dec = matches.groupby(['toss_decision', 'team1_won']).size().unstack()
toss_dec.plot(kind='bar', stacked=True)
plt.title('Toss Decision vs Match Outcome')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Model Performance Comparison

In [ ]:
results_path = os.path.join(RESULTS_DIR, 'model_results.json')
with open(results_path) as f:
    results = json.load(f)

df_res = pd.DataFrame(results).T
df_res[['cv_accuracy', 'test_accuracy', 'test_roc_auc']].plot(kind='bar', figsize=(10,6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.1)
plt.xticks(rotation=30)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random Baseline')
plt.legend()
plt.tight_layout()
plt.show()

df_res

## 7. IPL 2026 Prediction Results

In [ ]:
pred_path = os.path.join(RESULTS_DIR, 'prediction_2026.json')
with open(pred_path) as f:
    pred_data = json.load(f)

rankings = pred_data['rankings']
df_pred = pd.DataFrame(rankings).set_index('rank')

PALETTE = {
    'CSK': '#F9CD02', 'MI': '#004BA0', 'RCB': '#EC1C24', 'KKR': '#3A225D',
    'DC': '#00008B', 'PBKS': '#ED1B24', 'RR': '#2D9CDB', 'SRH': '#F7A721',
    'LSG': '#A2D9CE', 'GT': '#1B2A4A',
}

fig, ax = plt.subplots(figsize=(12, 7))
teams = [r['team_name'] for r in rankings]
probs = [r['win_probability'] for r in rankings]
colors = [PALETTE.get(r['team_id'], '#888') for r in rankings]

bars = ax.barh(teams[::-1], probs[::-1], color=colors[::-1], edgecolor='white', height=0.6)
for bar, prob in zip(bars, probs[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{prob:.2f}%', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Win Probability (%)', fontsize=13)
ax.set_title('IPL 2026 Champion Prediction\n(Stacking Ensemble + Bayesian Prior Update)',
             fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f"\n🏆 Predicted IPL 2026 Winner: {rankings[0]['team_name']}")
print(f"   Win probability: {rankings[0]['win_probability']:.2f}%")